# WATCHDOG Modbus Client Dokumentation

Dieses Notebook dokumentiert die Datei `watchdog/modbus_client.py`. Das Modul kapselt die gesamte Modbus-RTU-Kommunikation mit dem angeschlossenen Gerät.

## Zweck

Der `WatchdogModbusClient` stellt eine einheitliche Schnittstelle für:

- Verbindungsaufbau zum Modbus-Gerät
- Lesen von Holding-Registern
- Lesen von Input-Registern
- Skalierung von Rohwerten
- Vereinheitlichung unterschiedlicher PyModbus-Versionen
- Bereitstellung strukturierter Messdaten


In [ ]:
from watchdog.modbus_client import WatchdogModbusClient

client = WatchdogModbusClient()

## Klassenübersicht

### WatchdogModbusClient

Zentrale Kommunikationsklasse für Modbus RTU.

**Methoden**
- `connect()`
- `close()`
- `_call_with_slave_id()`
- `_read_holding_registers()`
- `_read_input_registers()`
- `read_register()`
- `read_all_registers()`

## Initialisierung

Beim Erzeugen der Klasse wird ein `ModbusSerialClient` anhand der Werte aus `MODBUS_CONFIG` konfiguriert.

Verwendete Parameter:

- Port
- Baudrate
- Parität
- Stopbits
- Bytesize
- Timeout
- Slave-ID

## Kompatibilität zu PyModbus

Die Methode `_call_with_slave_id()` erkennt automatisch, welche Parameterbezeichnung die installierte PyModbus-Version erwartet.

Unterstützte Varianten:

```python
slave=...
unit=...
device_id=...
```

Dadurch bleibt der Code kompatibel mit verschiedenen PyModbus-Releases.

## Registerzugriff

### Holding Register

```python
_read_holding_registers()
```

Verwendet `read_holding_registers()`.

### Input Register

```python
_read_input_registers()
```

Verwendet `read_input_registers()`.

## Messwertverarbeitung

Die Methode `read_register()` führt folgende Schritte aus:

1. Registerdefinition auswerten
2. Register lesen
3. Fehler prüfen
4. Rohwert extrahieren
5. Skalierung anwenden
6. Ergebnis als Dictionary zurückgeben

Beispielstruktur:

```python
{
    "name": "water_temp",
    "raw_value": 215,
    "value": 21.5,
    "unit": "°C",
    "description": "Wassertemperatur",
    "address": 100,
    "type": "input"
}
```

## REGISTER_MAP

Alle auszulesenden Register werden zentral in `register_map.py` definiert.

```text
REGISTER_MAP
    │
    ├── Temperatur
    ├── Druck
    ├── Status
    └── Betriebswerte
```

Neue Register können ohne Änderungen am Modbus-Client ergänzt werden.

## Datenfluss

```text
REGISTER_MAP
      │
      ▼
read_all_registers()
      │
      ▼
read_register()
      │
      ▼
Modbus Gerät
      │
      ▼
Dictionary mit Messwerten
      │
      ▼
main.py
      │
      ▼
SQLite Datenbank
```

## Fehlerbehandlung

Mögliche Fehlerfälle:

- Ungültiger Registertyp
- Kommunikationsfehler
- Timeout
- Fehlerhafte Antwort des Gerätes

Bei Fehlern wird eine Exception erzeugt und an die Anwendung weitergegeben.

## ADR-004: Zentrale Modbus-Abstraktion

### Status
Accepted

### Entscheidung
Die Modbus-Kommunikation wird vollständig in einer eigenen Klasse gekapselt.

### Vorteile
- Trennung von Geschäftslogik und Kommunikation
- Einfache Testbarkeit
- Austausch der Kommunikationsbibliothek möglich
- Wiederverwendbarkeit

### Nachteile
- Zusätzliche Abstraktionsschicht

### Zukunft
- Unterstützung mehrerer Geräte
- 32-Bit Register
- Float-Dekodierung
- Batch-Leseoperationen
- Retry-Strategien
- Kommunikationsstatistiken

## Verbesserungsvorschläge

1. Unterstützung für Signed Integer
2. Unterstützung für Float32/Float64
3. Register-Batching zur Performance-Steigerung
4. Retry-Mechanismus bei Timeouts
5. Kommunikationsmetriken für Monitoring
6. Unit-Tests mit simuliertem Modbus-Server